In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent)

print(Path.cwd())

d:\private\ai-research-paper-assistant


In [2]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from src.ingestion.parser import SimpleLoader
from src.chunker import RecursiveChunk
from src.embedder import OllamaEmbedder
from src.database import PGVectorStore
from src.generation import OllamaLLM, prompt
from src.reranker import CrossEncoderReRanker
from src.utils.helpers import format_context, IngestionPipeline
from src.retriever import HybridRetriever, DenseRetriever

c:\Users\luann\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_one_json(path: str):
    path = Path(path)
    if not path.exists():
        raise ValueError("Invalid path")
    with open(path, "r", encoding="utf-8") as f:
        contents = json.load(f)
    return contents

def load_dir_json(dir_path: str):
    path = Path(dir_path)
    if not path.exists():
        raise ValueError("Invalid path")
    all_files = path.glob("*.json")
    all_contents = []
    for file in tqdm(all_files, desc="Processing"):
        all_contents.append(load_one_json(file))
    return all_contents

In [4]:
all_contents = load_dir_json(r"D:\private\ai-research-paper-assistant\data\eval")
print(len(all_contents))
print(all_contents[:10])

Processing: 5it [00:00, 514.18it/s]

5
[{'dataset_name': 'arcface_additive_angular_margin_loss_rag_eval', 'samples': [{'id': 'q001', 'difficulty': 'Easy', 'question_type': 'Definition', 'question': 'What is the geometric interpretation of the additive angular margin penalty in ArcFace?', 'reference_answer': 'The additive angular margin penalty in ArcFace corresponds directly to the geodesic distance margin penalty on the normalized hypersphere.', 'evidence': [{'page': 3, 'section': '3.1 ArcFace', 'evidence_text': 'Since the proposed additive angular margin penalty is equal to the geodesic distance margin penalty in the normalized hypersphere, we name our method as ArcFace.'}]}, {'id': 'q002', 'difficulty': 'Easy', 'question_type': 'Architecture', 'question': 'What specific layer structure is applied after the final convolutional layer to generate the 512-D embedding feature in ArcFace?', 'reference_answer': 'After the last convolutional layer, the architecture utilizes a BN-Dropout-FC-BN structure to produce the final 512

## Initating

In [5]:
loader = SimpleLoader()
chunker = RecursiveChunk(chunk_size=1200, chunk_overlap=50)
embedder = OllamaEmbedder(dimensions=1024, num_gpu=1, num_ctx=4096) #num_ctx=2048
repo = PGVectorStore()
llm = OllamaLLM(temperature=0.0, model_name="qwen3.5:2b", num_gpu=1, reasoning=False, num_ctx=4096, num_predict=1024)#num_ctx=2048
reranker = CrossEncoderReRanker()

ingestor = IngestionPipeline(loader=loader, chunker=chunker, embedder=embedder, repository=repo)

## Ingesting data into DB

In [ ]:
repo.delete()
repo.count()

251

In [7]:
papers_path = r"D:\private\ai-research-paper-assistant\papers"
ingestor.ingest_dir(papers_path)

Loading PDFs: 0it [00:00, ?it/s][INFO] 2026-10-03 17:46:13,071 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-03 17:46:13,184 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\luann\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-03 17:46:13,185 [RapidOCR] main.py:63: Using C:\Users\luann\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-03 17:46:13,277 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-03 17:46:13,281 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\luann\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-03 17:46:13,281 [RapidOCR] main.py:63: Using C:\Users\luann\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-03 17:46:13,341 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-03

Inserted 251 chunks from papers folder.


251

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3.5-2B"
)
def create_eval_data(mode="hybrid", rerank=False, candidate_k=5, top_k=5):
    if mode == "hybrid":
        retriever = HybridRetriever(embedder, repo)
    else:
        retriever = DenseRetriever(embedder, repo)
    if rerank:
        reranker = CrossEncoderReRanker(device="cuda")
    
    max_tokens = float("-inf")
    for question, reference in tqdm(zip(eval_questions, eval_answers), desc="Processing", total=len(eval_questions)):
        context_retrieved = retriever.retrieve(question, top_k=candidate_k)
        if rerank:
            context_retrieved = reranker.rerank(question, context_retrieved, top_k=top_k)
        # RAGAS cần list[str]
        retrieved_contexts = [doc["content"] for doc in context_retrieved]
         # LLM của RAG cần formatted string
        context = format_context(context_retrieved)
        message = prompt.invoke({
                             "context": context,
                             "question": question
                             })
        tokens = tokenizer.encode(message.to_string(), add_special_tokens=True)

        max_tokens = max(max_tokens, len(tokens))
    print(max_tokens)

In [7]:
all_chunks = repo.get_all_chunks()
cnt = 0
for chunk in all_chunks:
    print(chunk["metadata"].get("source", ""))

D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.02640v5.pdf
D:\private\ai-research-paper-assistant\papers\1506.0264

## Creating data for evaluating

In [8]:
# eval_questions = [
#         "Can you provide a concise description of the TinyLlama model?",
#         "I would like to know the speed optimizations that TinyLlama has made.",
#         "Why TinyLlama uses Grouped-query Attention?",
#         "Is the TinyLlama model open source?",
#         "Tell me about starcoderdata dataset",
#     ]

# eval_answers = [
#         "TinyLlama is a compact 1.1B language model pretrained on around 1 trillion tokens for approximately 3 epochs. Building on the architecture and tokenizer of Llama 2, TinyLlama leverages various advances contributed by the open-source community (e.g., FlashAttention), achieving better computational efficiency. Despite its relatively small size, TinyLlama demonstrates remarkable performance in a series of downstream tasks. It significantly outperforms existing open-source language models with comparable sizes.",
#         "During training, our codebase has integrated FSDP to leverage multi-GPU and multi-node setups efficiently. Another critical improvement is the integration of Flash Attention, an optimized attention mechanism. We have replaced the fused SwiGLU module from the xFormers (Lefaudeux et al., 2022) repository with the original SwiGLU module, further enhancing the efficiency of our codebase. With these features, we can reduce the memory footprint, enabling the 1.1B model to fit within 40GB of GPU RAM.",
#         "To reduce memory bandwidth overhead and speed up inference, we use grouped-query attention in our model. We have 32 heads for query attention and use 4 groups of key-value heads. With this technique, the model can share key and value representations across multiple heads without sacrificing much performance",
#         "Yes, TinyLlama is open-source",
#         "This dataset was collected to train StarCoder (Li et al., 2023), a powerful opensource large code language model. It comprises approximately 250 billion tokens across 86 programming languages. In addition to code, it also includes GitHub issues and text-code pairs that involve natural languages.",
#     ]

eval_questions = []
eval_answers = []
sampels = []

for doc in all_contents:
    sample = doc["samples"]
    df = pd.DataFrame(sample)
    easy_q = df[df["difficulty"]=="Easy"].sample(7)
    medium_q = df[df["difficulty"]=="Medium"].sample(7)
    hard_q = df[df["difficulty"]=="Hard"].sample(6)
    selected = pd.concat([easy_q, medium_q, hard_q])
    eval_questions.extend(selected["question"].to_list())
    eval_answers.extend(selected["reference_answer"].to_list())
    
print(len(eval_questions))
print(len(eval_answers))

100
100


In [9]:
for q, ans in zip(eval_questions, eval_answers):
    print(q)
    print(ans)
    print("-"*50)

What 1:1 verification performance (TPR@FPR=1e-4) does the ArcFace ResNet100 model achieve on the IJB-C benchmark when trained on IBUG-500K?
ArcFace with ResNet100 trained on IBUG-500K achieves a TPR of 97.27% at FPR=1e-4 on the IJB-C benchmark.
--------------------------------------------------
What standard default values are chosen for the feature scale s and the angular margin m in the ArcFace experiments?
The feature scale s is set to 64 and the additive angular margin m is set to 0.5.
--------------------------------------------------
In sub-center ArcFace, how is the class-wise similarity score derived from the subclass-wise similarity scores?
The class-wise similarity score S' is computed by applying a max pooling operation across the K subclass-wise similarity scores for each class.
--------------------------------------------------
What specific layer structure is applied after the final convolutional layer to generate the 512-D embedding feature in ArcFace?
After the last con

In [10]:
all_chunks = repo.get_all_chunks()
all_chunks[0]

{'id': 38312,
 'document_id': '1506.02640v5',
 'content': 'You Only Look Once:\nUniﬁed, Real-Time Object Detection\nJoseph Redmon∗, Santosh Divvala∗†, Ross Girshick¶, Ali Farhadi∗†\nUniversity of Washington∗, Allen Institute for AI†, Facebook AI Research¶\nhttp://pjreddie.com/yolo/\nAbstract\nWe present YOLO, a new approach to object detection.\nPrior work on object detection repurposes classiﬁers to per-\nform detection. Instead, we frame object detection as a re-\ngression problem to spatially separated bounding boxes and\nassociated class probabilities. A single neural network pre-\ndicts bounding boxes and class probabilities directly from\nfull images in one evaluation. Since the whole detection\npipeline is a single network, it can be optimized end-to-end\ndirectly on detection performance.\nOur uniﬁed architecture is extremely fast. Our base\nYOLO model processes images in real-time at 45 frames\nper second. A smaller version of the network, Fast YOLO,\nprocesses an astounding 1

In [11]:
def create_eval_data(mode="hybrid", rerank=False, candidate_k=5, top_k=5):
    evaluation_data = []
    if mode == "hybrid":
        retriever = HybridRetriever(embedder, repo)
        retriever.fit()
    else:
        retriever = DenseRetriever(embedder, repo)
    if rerank:
        reranker = CrossEncoderReRanker(device="cuda")
    

    for question, reference in tqdm(zip(eval_questions, eval_answers), desc="Processing", total=len(eval_questions)):
        context_retrieved = retriever.retrieve(question, top_k=candidate_k)
        if rerank:
            context_retrieved = reranker.rerank(question, context_retrieved, top_k=top_k)
        # RAGAS cần list[str]
        retrieved_contexts = [doc["content"] for doc in context_retrieved]
         # LLM của RAG cần formatted string
        context = format_context(context_retrieved)

        message = prompt.invoke({
            "context": context,
            "question": question
            })

        response = llm.generate(
            message
            ).content

        evaluation_data.append({
            "user_input": question,
            "reference": reference,
            "retrieved_contexts": retrieved_contexts,
            "response": response,
            })
    return evaluation_data
    # Query/user_input, predict_chunk_ids(document_id), reference_chunk_ids(document_id), reference, retrieved_contexts, response

In [ ]:
eval_data_hybrid = create_eval_data(mode="hybrid", candidate_k=5)

Processing: 100%|██████████| 100/100 [36:17<00:00, 21.78s/it]


In [12]:
eval_data_hybrid_rerank = create_eval_data(mode="hybrid", rerank=True, candidate_k=50, top_k=5)

Processing: 100%|██████████| 100/100 [33:35<00:00, 20.15s/it]


In [ ]:
eval_data_dense = create_eval_data(mode="dense", candidate_k=5)

Processing: 100%|██████████| 100/100 [36:48<00:00, 22.08s/it]


In [13]:
eval_data_dense_rerank = create_eval_data(mode="dense", rerank=True, candidate_k=50, top_k=5)

Processing: 100%|██████████| 100/100 [34:12<00:00, 20.53s/it]


In [19]:
import json
with open(r"D:\private\ai-research-paper-assistant\data\eval_data_dense_rerank.json", "r", encoding="utf-8") as f:
    eval_data_hybrid_rerank = json.load(f)

print(eval_data_hybrid_rerank[0]["user_input"])
print()

print("REFERENCE:")
print(eval_data_hybrid_rerank[0]["reference"])
print()

print("CONTEXTS:")
print(eval_data_hybrid_rerank[0]["retrieved_contexts"])
print(type(eval_data_hybrid_rerank[0]["retrieved_contexts"]))
print(len(eval_data_hybrid_rerank[0]["retrieved_contexts"]))

print("\nRESPONSE:")
print(eval_data_hybrid_rerank[0]["response"])

What 1:1 verification performance (TPR@FPR=1e-4) does the ArcFace ResNet100 model achieve on the IJB-C benchmark when trained on IBUG-500K?

REFERENCE:
ArcFace with ResNet100 trained on IBUG-500K achieves a TPR of 97.27% at FPR=1e-4 on the IJB-C benchmark.

CONTEXTS:
['(a) ROC for IJB-B\n10 6\n10 5\n10 4\n10 3\n10 2\n10 1\nFalse Positive Rate\n0.70\n0.75\n0.80\n0.85\n0.90\n0.95\n1.00True Positive Rate\nVGG2\nMS1MV3\nIBUG500K (b) ROC for IJB-C\nFig. 12. ROC curves of 1:1 veriﬁcation protocol on IJB-B and IJB-C.\n([Dataset*, ResNet100, ArcFace])\nthe training data and the ResNet50 as the embedding network\nto train ArcFace for the fair comparison with the most recent\nsoftmax-based methods [9], [113], [94]. As we can see from\nthe results, the proposed additive angular margin can obviously\nboost the performance on both IJB-B and IJB-C compared to the\nsoftmax loss (about 3∼ 5%, which is a signiﬁcant reduction in\nthe error).\nDrawing support from more training data (IBUG-500K) and\ndeep

## Save file

In [14]:
import json

def export(sp, file):
    with open(sp, "w", encoding="utf-8") as f:
        json.dump(file, f)
base_path = "data"
# export(os.path.join(base_path, "eval_data_dense.json"), eval_data_dense)
# export(os.path.join(base_path, "eval_data_hybrid.json"), eval_data_hybrid)
export(os.path.join(base_path, "eval_data_dense_rerank.json"), eval_data_dense_rerank)
export(os.path.join(base_path, "eval_data_hybrid_rerank.json"), eval_data_hybrid_rerank)